# Multi-Layer Automotive Intrusion Detection System (IDS)
### Combining CAN Bus Anomaly Detection and Telematics Behavioral Monitoring

This notebook implements a complete automotive IDS as per the HCRL dataset requirements. It covers:
1. **CAN Bus Parsing & Exploration**
2. **Feature Engineering** (Timing, Entropy, N-grams)
3. **Model Development** (XGBoost, Isolation Forest)
4. **Telematics Derivation** (Speed, RPM, Acceleration)
5. **Multi-Layer Correlation** (Integrated vSOC Alerts)
6. **Deployment Analysis** (Resource Profiling)

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add src to path
sys.path.append(os.getcwd())

from src.parser import CANParser
from src.features import CANFeatureEngineer
from src.validation import DataSplitter
from src.models import CANModelTrainer
from src.evaluation import IDSEvaluator
from src.telematics import TelematicsProcessor, BehavioralBaseline
from src.integration import AlertCorrelator
from src.deployment import ModelProfiler, LightweightModelTrainer

print("Setup Complete. Dependencies Loaded.")

## Part 1: CAN Bus Intrusion Detection
We start by parsing a sample of the **DoS Attack** dataset and extracting message-level features.

In [ ]:
# Load and Parse Data
file_path = 'Data/DoS_dataset.csv'
chunk_gen = CANParser.load_csv(file_path, chunksize=20000)
df = next(chunk_gen)
df = CANParser.preprocess_df(df)

# Feature Engineering
print("Extracting features...")
df = CANFeatureEngineer.extract_message_features(df)
df = CANFeatureEngineer.extract_window_features(df, window_size=50)

print(f"Data Sample (Processed):\n{df.head()}")

### Attack Visualization
Let's visualize the timing intervals (Delta T) for the DoS attack. Injection attacks typically show extreme drops in Delta T for the targeted ID (e.g., ID `0000`).

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=df.iloc[1000:2000], x='Timestamp', y='Delta_T', hue='Flag')
plt.title("CAN Message Timing Intervals (Normal vs. Attack)")
plt.ylabel("Delta T (seconds)")
plt.show()

### Model Training (XGBoost)
We use strict temporal splitting for validation as mandated.

In [ ]:
X, y = CANFeatureEngineer.get_feature_matrix(df)
X_train, X_test, y_train, y_test = DataSplitter.temporal_split(X, y, test_size=0.2)

trainer = CANModelTrainer(model_type='xgboost')
trainer.train(X_train, y_train)

y_pred = trainer.predict(X_test)
metrics = IDSEvaluator.calculate_metrics(y_test, y_pred)
latency = IDSEvaluator.measure_latency(trainer.model, X_test)

print(f"Model Performance: {metrics}")
print(f"Inference Latency: {latency:.2f} ms")

## Part 2: Telematics Behavioral Monitoring
We derive vehicle behavior (Speed, RPM) from the CAN signals and establish a baseline.

In [ ]:
# Use normal run data for baseline
norm_df = CANParser.parse_raw_text('Data/normal_run_data/normal_run_data.txt', max_lines=10000)
norm_df = CANParser.preprocess_df(norm_df)
tele_df = TelematicsProcessor.derive_features(norm_df)

baseline = BehavioralBaseline()
baseline.fit(tele_df)

plt.figure(figsize=(10, 4))
plt.plot(tele_df['Timestamp'], tele_df['Speed'], label='Vehicle Speed (Proxy)')
plt.title("Normal Driving Behavioral Baseline (Speed)")
plt.legend()
plt.show()

## Part 3: Multi-Layer Correlation & vSOC Alerts
Integrating CAN-level alerts with telematics anomalies to generate high-confidence vSOC reports.

In [ ]:
# Mock alerts for demonstration
can_alerts = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Attack_Type': ['DoS']})
tele_anomalies = pd.DataFrame({'Timestamp': [df['Timestamp'].iloc[-1]], 'Anomaly_Score': [1]})

correlated = AlertCorrelator.correlate(can_alerts, tele_anomalies)
vsoc_alert = AlertCorrelator.generate_vsoc_alert(correlated.iloc[0])

print("--- Generated vSOC Alert ---")
print(json.dumps(vsoc_alert, indent=2))

## Part 4: Deployment Analysis
Comparing heavy (XGBoost) and light (Decision Tree) models for ECU resource constraints.

In [ ]:
light_trainer = LightweightModelTrainer(max_depth=3)
light_trainer.train(X_train, y_train)

heavy_stats = ModelProfiler.profile_inference(trainer.model, X_test)
light_stats = ModelProfiler.profile_inference(light_trainer.model, X_test)

print(f"Heavy Model Throughput: {heavy_stats['throughput_mps']:.0f} MPS")
print(f"Light Model Throughput: {light_stats['throughput_mps']:.0f} MPS")